In [ ]:
!pip install -q roboflow ultralytics pandas numpy pyyaml opencv-python matplotlib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from getpass import getpass

from pathlib import Path
import os
import random
import shutil
import yaml
import json
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

from roboflow import Roboflow
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# Roboflow project settings
# =========================
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
ROBOFLOW_WORKSPACE = "bhavya-johar"
ROBOFLOW_PROJECT = "final-dataset-yz6ya"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolo26"

# =========================
# Local paths
# =========================
DATASET_DIR = Path("/content/full_galvat_trial5_yolo26")
BACKGROUND_SOURCE_DIR = Path("/content/background_source_images")  # put candidate background images here
BACKGROUND_CROP_DIR = Path("/content/background_crops_1500")

# Training settings
MODEL_SIZE = "yolo26s.pt"   # alternatives: yolo26n.pt, yolo26m.pt, yolo26l.pt, yolo26x.pt
IMG_SIZE = 640
EPOCHS = 100
BATCH = 16
PATIENCE = 25
RUN_NAME = "full_galvat_trial5_yolo26s_640"

In [ ]:
# =========================
# Cell 4: Download Roboflow ZIP directly from export link
# =========================

from pathlib import Path
import shutil
import zipfile
import subprocess

# Paste your real Roboflow ds link here
ROBOFLOW_ZIP_URL = "https://app.roboflow.com/ds/9iFrgFlf3Z?key=ipZf4I5oZu"
RAW_DIR = Path("/content/final_dataset_raw")
PROCESSED_DIR = Path("/content/final_dataset_processed_640")
ZIP_PATH = Path("/content/roboflow.zip")

# Clean old files
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download zip
print("Downloading Roboflow ZIP...")
subprocess.run(
    ["curl", "-L", ROBOFLOW_ZIP_URL, "-o", str(ZIP_PATH)],
    check=True
)

print("Downloaded to:", ZIP_PATH)
print("ZIP size MB:", ZIP_PATH.stat().st_size / 1_000_000)

# Unzip
print("Unzipping...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(RAW_DIR)

# Optional: remove zip after extraction to save space
ZIP_PATH.unlink()

DATASET_DIR = RAW_DIR

print("Unzipped raw dataset to:", RAW_DIR)
print("\nTop-level files/folders:")
for p in RAW_DIR.iterdir():
    print("-", p.name)

In [ ]:
# =========================
# Cell 5: Find data.yaml and inspect classes
# =========================

yaml_files = list(DATASET_DIR.rglob("data.yaml")) + list(DATASET_DIR.rglob("data.yml"))

if not yaml_files:
    raise FileNotFoundError("No data.yaml or data.yml found in the downloaded ZIP.")

DATA_YAML_RAW = yaml_files[0]

with open(DATA_YAML_RAW, "r") as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]

print("Using data.yaml:", DATA_YAML_RAW)
print("Class count:", len(names))

for i, name in enumerate(names):
    print(i, name)

In [ ]:
# =========================
# Cell 6: Auto-orient + resize all images to 640x640
# =========================

from PIL import Image, ImageOps

RAW_DIR = DATASET_DIR
PROCESSED_DIR = Path("/content/final_dataset_processed_640")

if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

IMG_SIZE = 640
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_split_dirs(root, split):
    aliases = {
        "train": ["train"],
        "valid": ["valid", "val", "validation"],
        "test": ["test"],
    }

    for alias in aliases[split]:
        img_dir = root / alias / "images"
        lbl_dir = root / alias / "labels"

        if img_dir.exists() and lbl_dir.exists():
            return alias, img_dir, lbl_dir

    raise FileNotFoundError(f"Could not find {split} images/labels folders.")

def process_image(src_path, dst_path, size=640):
    with Image.open(src_path) as img:
        # Auto-orient from EXIF
        img = ImageOps.exif_transpose(img)

        # Convert all images to RGB
        img = img.convert("RGB")

        # Stretch resize to 640x640
        # YOLO normalized labels stay valid under full-image stretch resize
        img = img.resize((size, size), Image.Resampling.BILINEAR)

        img.save(dst_path, quality=95)

for split in ["train", "valid", "test"]:
    try:
        src_split_name, src_img_dir, src_lbl_dir = find_split_dirs(RAW_DIR, split)
    except FileNotFoundError:
        print(f"Skipping missing split: {split}")
        continue

    dst_split = "valid" if split == "valid" else split
    dst_img_dir = PROCESSED_DIR / dst_split / "images"
    dst_lbl_dir = PROCESSED_DIR / dst_split / "labels"

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = [
        p for p in src_img_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    processed = 0
    skipped = 0

    for img_path in images:
        rel = img_path.relative_to(src_img_dir)
        stem = rel.with_suffix("").as_posix().replace("/", "_")

        dst_img = dst_img_dir / f"{stem}.jpg"
        src_lbl = src_lbl_dir / rel.with_suffix(".txt")
        dst_lbl = dst_lbl_dir / f"{stem}.txt"

        try:
            process_image(img_path, dst_img, IMG_SIZE)

            if src_lbl.exists():
                shutil.copy2(src_lbl, dst_lbl)
            else:
                # Empty label file for background/negative image
                dst_lbl.write_text("")

            processed += 1

        except Exception as e:
            print("Failed:", img_path, e)
            skipped += 1

    print(f"{split}: processed {processed}, skipped {skipped}")

print("\nProcessed dataset saved to:", PROCESSED_DIR)

In [ ]:
# =========================
# Cell 7: Write clean processed data.yaml
# =========================

DATA_YAML_PROCESSED = PROCESSED_DIR / "data.yaml"

data_yaml = {
    "path": str(PROCESSED_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(names),
    "names": names,
}

with open(DATA_YAML_PROCESSED, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(DATA_YAML_PROCESSED.read_text())

In [ ]:
# =========================
# Cell 7B: Clean labels to detection-only format before verification/background ROI crops
# =========================
# This must run BEFORE dataset verification and BEFORE background ROI crop generation.
# For YOLO object detection, each label row must be exactly:
# class_id x_center y_center width height

from pathlib import Path

PROCESSED_DIR = Path(PROCESSED_DIR)

fixed_files = 0
bad_rows = 0
total_rows = 0

for label_file in PROCESSED_DIR.rglob("*.txt"):
    if "/labels/" not in str(label_file):
        continue

    old_lines = label_file.read_text().splitlines()
    new_lines = []

    for line in old_lines:
        parts = line.strip().split()

        # Empty label file is valid for background/no-object images
        if not parts:
            continue

        if len(parts) < 5:
            bad_rows += 1
            continue

        # Keep only detection columns:
        # class x_center y_center width height
        new_lines.append(" ".join(parts[:5]))
        total_rows += 1

    old_text = "\n".join(old_lines).strip()
    new_text = "\n".join(new_lines).strip()

    if old_text != new_text:
        label_file.write_text(new_text + ("\n" if new_text else ""))
        fixed_files += 1

# Delete YOLO cache files so Ultralytics rescans clean labels
deleted_caches = []
for cache_file in PROCESSED_DIR.rglob("*.cache"):
    cache_file.unlink()
    deleted_caches.append(str(cache_file))

print("Fixed label files:", fixed_files)
print("Bad rows skipped:", bad_rows)
print("Total rows kept:", total_rows)
print("Deleted cache files:", len(deleted_caches))

In [ ]:
# =========================
# Cell 8: Verify processed dataset
# =========================

import cv2
import pandas as pd
from collections import Counter
from pathlib import Path

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def verify_processed_dataset(root, class_names):
    root = Path(root)
    rows = []
    class_counts_by_split = {}

    for split in ["train", "valid", "test"]:
        img_dir = root / split / "images"
        lbl_dir = root / split / "labels"

        if not img_dir.exists():
            print(f"\n{split}: MISSING images folder")
            continue

        if not lbl_dir.exists():
            print(f"\n{split}: MISSING labels folder")
            continue

        images = [
            p for p in img_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        ]

        labels = [
            p for p in lbl_dir.rglob("*.txt")
            if p.is_file()
        ]

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        missing_labels = sorted(image_stems - label_stems)
        orphan_labels = sorted(label_stems - image_stems)

        size_counter = Counter()
        class_counter = Counter()
        bad_rows = []

        for img_path in images:
            img = cv2.imread(str(img_path))
            if img is not None:
                h, w = img.shape[:2]
                size_counter[(w, h)] += 1

        for lbl_path in labels:
            for line_num, line in enumerate(lbl_path.read_text().splitlines(), start=1):
                parts = line.strip().split()

                # Empty label file is valid for background/negative images
                if not parts:
                    continue

                if len(parts) < 5:
                    bad_rows.append((str(lbl_path), line_num, "too_few_parts"))
                    continue

                try:
                    cls_id = int(float(parts[0]))
                    coords = [float(x) for x in parts[1:5]]
                except Exception:
                    bad_rows.append((str(lbl_path), line_num, "parse_error"))
                    continue

                if cls_id < 0 or cls_id >= len(class_names):
                    bad_rows.append((str(lbl_path), line_num, f"bad_class_id_{cls_id}"))
                    continue

                if any(c < 0 or c > 1 for c in coords):
                    bad_rows.append((str(lbl_path), line_num, f"coords_out_of_range_{coords}"))

                class_counter[class_names[cls_id]] += 1

        class_counts_by_split[split] = class_counter

        rows.append({
            "split": split,
            "images": len(images),
            "labels": len(labels),
            "missing_labels": len(missing_labels),
            "orphan_labels": len(orphan_labels),
            "bad_label_rows": len(bad_rows),
            "most_common_image_sizes": size_counter.most_common(5),
        })

        print("\n" + "=" * 70)
        print(split.upper())
        print("=" * 70)
        print("Images:", len(images))
        print("Labels:", len(labels))
        print("Missing labels:", len(missing_labels))
        print("Orphan labels:", len(orphan_labels))
        print("Bad label rows:", len(bad_rows))
        print("Image sizes:", size_counter.most_common(5))

        if missing_labels:
            print("Example missing labels:", missing_labels[:10])
        if orphan_labels:
            print("Example orphan labels:", orphan_labels[:10])
        if bad_rows:
            print("Example bad label rows:", bad_rows[:10])

    verify_df = pd.DataFrame(rows)
    return verify_df, class_counts_by_split

verify_df, class_counts_by_split = verify_processed_dataset(PROCESSED_DIR, names)
display(verify_df)

# Stop if there are serious dataset issues
if len(verify_df) == 0:
    raise RuntimeError("No valid splits found.")

if verify_df["bad_label_rows"].sum() > 0:
    raise RuntimeError("Bad label rows found. Fix labels before training.")

if verify_df["missing_labels"].sum() > 0:
    print("WARNING: Some images are missing labels. Empty labels are allowed, but only if intentional.")

if verify_df["orphan_labels"].sum() > 0:
    print("WARNING: Some labels have no matching images.")

In [ ]:
# =========================
# Cell 9: Create 1,500 background ROI crops from existing dataset images
# =========================

import random
import cv2
import shutil
import numpy as np
from pathlib import Path

N_BACKGROUND_CROPS = 1500
ROI_MIN_SIZE = 160          # smallest random crop side before resizing to 640
ROI_MAX_SIZE = 420          # largest random crop side before resizing to 640
TARGET_SIZE = 640
MAX_ATTEMPTS = N_BACKGROUND_CROPS * 100
MIN_DISTANCE_FROM_OBJECT = 0.02  # small safety margin around boxes, normalized
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

BACKGROUND_CROP_DIR = Path("/content/background_roi_crops_from_dataset")

if BACKGROUND_CROP_DIR.exists():
    shutil.rmtree(BACKGROUND_CROP_DIR)

(BACKGROUND_CROP_DIR / "images").mkdir(parents=True, exist_ok=True)
(BACKGROUND_CROP_DIR / "labels").mkdir(parents=True, exist_ok=True)

train_img_dir = PROCESSED_DIR / "train" / "images"
train_lbl_dir = PROCESSED_DIR / "train" / "labels"

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

train_images = [
    p for p in train_img_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
]

print("Training images available:", len(train_images))

def yolo_to_xyxy(line):
    """
    Converts YOLO normalized label line to xyxy normalized box.
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    try:
        cls = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:5])
    except Exception:
        return None

    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2

    return [x1, y1, x2, y2]

def load_boxes_for_image(img_path):
    """
    Loads normalized xyxy boxes for an image from its matching YOLO label file.
    """
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"

    if not lbl_path.exists():
        return []

    boxes = []

    for line in lbl_path.read_text().splitlines():
        box = yolo_to_xyxy(line)
        if box is not None:
            boxes.append(box)

    return boxes

def expand_box(box, margin):
    x1, y1, x2, y2 = box
    return [
        max(0.0, x1 - margin),
        max(0.0, y1 - margin),
        min(1.0, x2 + margin),
        min(1.0, y2 + margin),
    ]

def boxes_intersect(a, b):
    """
    a and b are normalized xyxy boxes.
    """
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    return inter_x2 > inter_x1 and inter_y2 > inter_y1

def crop_intersects_any_object(crop_box, object_boxes, margin=0.02):
    """
    Returns True if crop overlaps any object box, with a small margin.
    """
    for obj_box in object_boxes:
        expanded = expand_box(obj_box, margin)
        if boxes_intersect(crop_box, expanded):
            return True
    return False

def random_crop_box(w, h, min_size, max_size):
    """
    Returns pixel crop and normalized crop.
    """
    crop_w = random.randint(min_size, min(max_size, w))
    crop_h = random.randint(min_size, min(max_size, h))

    if crop_w >= w or crop_h >= h:
        return None

    x1 = random.randint(0, w - crop_w)
    y1 = random.randint(0, h - crop_h)
    x2 = x1 + crop_w
    y2 = y1 + crop_h

    crop_norm = [x1 / w, y1 / h, x2 / w, y2 / h]

    return x1, y1, x2, y2, crop_norm

made = 0
attempts = 0
skipped_no_read = 0
skipped_overlap = 0
skipped_bad_crop = 0

while made < N_BACKGROUND_CROPS and attempts < MAX_ATTEMPTS:
    attempts += 1

    img_path = random.choice(train_images)
    img = cv2.imread(str(img_path))

    if img is None:
        skipped_no_read += 1
        continue

    h, w = img.shape[:2]

    object_boxes = load_boxes_for_image(img_path)

    crop_result = random_crop_box(
        w=w,
        h=h,
        min_size=ROI_MIN_SIZE,
        max_size=ROI_MAX_SIZE
    )

    if crop_result is None:
        skipped_bad_crop += 1
        continue

    x1, y1, x2, y2, crop_norm = crop_result

    if crop_intersects_any_object(
        crop_norm,
        object_boxes,
        margin=MIN_DISTANCE_FROM_OBJECT
    ):
        skipped_overlap += 1
        continue

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        skipped_bad_crop += 1
        continue

    crop_resized = cv2.resize(
        crop,
        (TARGET_SIZE, TARGET_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    out_name = f"background_roi_{made:05d}.jpg"
    out_img_path = BACKGROUND_CROP_DIR / "images" / out_name
    out_lbl_path = BACKGROUND_CROP_DIR / "labels" / f"background_roi_{made:05d}.txt"

    cv2.imwrite(str(out_img_path), crop_resized)

    # Empty label file = valid YOLO background / no-object image
    out_lbl_path.write_text("")

    made += 1

print("Background ROI crop generation complete.")
print("Requested:", N_BACKGROUND_CROPS)
print("Created:", made)
print("Attempts:", attempts)
print("Skipped unreadable images:", skipped_no_read)
print("Skipped overlapping object:", skipped_overlap)
print("Skipped bad crop:", skipped_bad_crop)
print("Saved to:", BACKGROUND_CROP_DIR)

if made < N_BACKGROUND_CROPS:
    print("WARNING: Fewer than requested crops were created. Try lowering ROI_MIN_SIZE or MIN_DISTANCE_FROM_OBJECT.")

In [ ]:
# =========================
# Cell 9B: Add generated background ROI crops to train split
# =========================

bg_img_dir = BACKGROUND_CROP_DIR / "images"
bg_lbl_dir = BACKGROUND_CROP_DIR / "labels"

bg_imgs = sorted(bg_img_dir.glob("*.jpg"))
bg_lbls = sorted(bg_lbl_dir.glob("*.txt"))

train_img_dir = PROCESSED_DIR / "train" / "images"
train_lbl_dir = PROCESSED_DIR / "train" / "labels"

for img in bg_imgs:
    shutil.copy2(img, train_img_dir / img.name)

for lbl in bg_lbls:
    shutil.copy2(lbl, train_lbl_dir / lbl.name)

print("Added background ROI images:", len(bg_imgs))
print("Added empty background ROI labels:", len(bg_lbls))
print("Training images folder:", train_img_dir)
print("Training labels folder:", train_lbl_dir)

In [ ]:
# =========================
# Cell 10: Re-verify after adding background ROI crops
# =========================

verify_df, class_counts_by_split = verify_processed_dataset(PROCESSED_DIR, names)
display(verify_df)

train_row = verify_df[verify_df["split"] == "train"].iloc[0]

print("\nTraining split after background ROI crops:")
print("Images:", train_row["images"])
print("Labels:", train_row["labels"])
print("Missing labels:", train_row["missing_labels"])
print("Orphan labels:", train_row["orphan_labels"])
print("Bad label rows:", train_row["bad_label_rows"])

if train_row["bad_label_rows"] > 0:
    raise RuntimeError("Bad label rows found after background crop insertion.")

if train_row["orphan_labels"] > 0:
    print("WARNING: Some label files have no matching image.")

if train_row["missing_labels"] > 0:
    print("WARNING: Some images have no label file. This is okay only if intentional, but your background crops should have empty label files.")

In [ ]:
# =========================
# Cell 11: Run identity, output folders, and environment capture
# =========================

from __future__ import annotations

import csv
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import yaml
from ultralytics import YOLO

# Confirm GPU before training
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not visible to this Colab kernel. Go to Runtime → Change runtime type → GPU, then reconnect and rerun setup cells."
    )

print("GPU:", torch.cuda.get_device_name(0))

# -------- Run identity --------
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"galvat_trial5_yolo26s_44class_processed640_{RUN_TS}"

# Save everything to Drive in a new folder
DRIVE_RUN_BASE = Path("/content/drive/MyDrive/galvat_trial5_training_outputs")
RUN_ROOT = DRIVE_RUN_BASE / RUN_ID

RUN_DIRS = {
    "root": RUN_ROOT,
    "logs": RUN_ROOT / "logs",
    "metrics": RUN_ROOT / "metrics",
    "models": RUN_ROOT / "models",
    "exports": RUN_ROOT / "exports",
    "env": RUN_ROOT / "env",
    "config": RUN_ROOT / "config",
    "reports": RUN_ROOT / "reports",
    "snapshots": RUN_ROOT / "snapshots",
    "analytics": RUN_ROOT / "analytics",
}

for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# -------- Training configuration --------
TRAIN_CFG = {
    "model": "yolo26s.pt",
    "imgsz": 640,
    "epochs": 100,
    "batch": 16,
    "workers": 8,
    "optimizer": "Adam",
    "lr0": 0.001,
    "freeze": 10,
    "augment": False,
    "patience": 20,
    "seed": 1337,
    "device": 0,
    "project": str(RUN_DIRS["models"]),
    "name": "train",
    "exist_ok": True,
    "cache": False,
    "plots": True,
    "save": True,
}

print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("DATA_YAML_PROCESSED:", DATA_YAML_PROCESSED)
print("PROCESSED_DIR:", PROCESSED_DIR)

In [ ]:
# =========================
# Cell 12: Utility helpers and environment capture
# =========================

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def run_cmd(cmd: list[str]) -> dict:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return {
        "cmd": cmd,
        "returncode": proc.returncode,
        "stdout": proc.stdout[-20000:],
        "stderr": proc.stderr[-20000:],
    }


env_payload = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python_version": sys.version,
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

write_json(RUN_DIRS["env"] / "environment.json", env_payload)

if torch.cuda.is_available():
    write_json(RUN_DIRS["env"] / "nvidia_smi.json", run_cmd(["nvidia-smi"]))

write_json(RUN_DIRS["env"] / "pip_freeze.json", run_cmd([sys.executable, "-m", "pip", "freeze"]))

print("Environment artifacts written to:", RUN_DIRS["env"])

In [ ]:
# =========================
# Cell 13: Dataset snapshot
# =========================

DATA_YAML_PROCESSED = Path(DATA_YAML_PROCESSED)
PROCESSED_DIR = Path(PROCESSED_DIR)

if not DATA_YAML_PROCESSED.exists():
    raise FileNotFoundError(f"Missing processed data.yaml: {DATA_YAML_PROCESSED}")

data_yaml = yaml.safe_load(DATA_YAML_PROCESSED.read_text(encoding="utf-8"))

# names should already exist from earlier cells, but reload from yaml to be safe
dataset_names = data_yaml["names"]
if isinstance(dataset_names, dict):
    dataset_names = [dataset_names[k] for k in sorted(dataset_names.keys(), key=lambda x: int(x))]

names = dataset_names

dataset_snapshot = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "run_id": RUN_ID,
    "processed_dir": str(PROCESSED_DIR),
    "data_yaml": str(DATA_YAML_PROCESSED),
    "class_count": len(names),
    "classes": names,
    "train_images_dir": str(PROCESSED_DIR / "train" / "images"),
    "train_labels_dir": str(PROCESSED_DIR / "train" / "labels"),
    "valid_images_dir": str(PROCESSED_DIR / "valid" / "images"),
    "valid_labels_dir": str(PROCESSED_DIR / "valid" / "labels"),
    "test_images_dir": str(PROCESSED_DIR / "test" / "images"),
    "test_labels_dir": str(PROCESSED_DIR / "test" / "labels"),
}

write_json(RUN_DIRS["snapshots"] / "dataset_snapshot.json", dataset_snapshot)

# Save class list
(RUN_DIRS["config"] / "classes.txt").write_text(
    "\n".join(names),
    encoding="utf-8"
)

# Copy data.yaml
shutil.copy2(DATA_YAML_PROCESSED, RUN_DIRS["config"] / "data.yaml")

print("Dataset snapshot written:", RUN_DIRS["snapshots"] / "dataset_snapshot.json")
print("Class count:", len(names))

In [ ]:
# =========================
# Cell 14: Clean labels to detection-only format
# =========================

fixed_files = 0
bad_rows = 0
total_rows = 0

for label_file in PROCESSED_DIR.rglob("*.txt"):
    if "/labels/" not in str(label_file):
        continue

    old_lines = label_file.read_text().splitlines()
    new_lines = []

    for line in old_lines:
        parts = line.strip().split()

        if not parts:
            continue

        if len(parts) < 5:
            bad_rows += 1
            continue

        # Keep only class x y w h
        new_lines.append(" ".join(parts[:5]))
        total_rows += 1

    old_text = "\n".join(old_lines).strip()
    new_text = "\n".join(new_lines).strip()

    if old_text != new_text:
        label_file.write_text(new_text + ("\n" if new_text else ""))
        fixed_files += 1

# Delete YOLO cache files so clean labels are rescanned
deleted_caches = []
for cache_file in PROCESSED_DIR.rglob("*.cache"):
    cache_file.unlink()
    deleted_caches.append(str(cache_file))

cleanup_report = {
    "fixed_label_files": fixed_files,
    "bad_rows_skipped": bad_rows,
    "total_rows_kept": total_rows,
    "deleted_cache_files": deleted_caches,
}

write_json(RUN_DIRS["reports"] / "label_cleanup_report.json", cleanup_report)

print("Fixed label files:", fixed_files)
print("Bad rows skipped:", bad_rows)
print("Total rows kept:", total_rows)
print("Deleted cache files:", len(deleted_caches))

In [ ]:
# =========================
# Cell 15: Persist run config
# =========================

run_config_payload = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "processed_dir": str(PROCESSED_DIR),
    "data_yaml": str(DATA_YAML_PROCESSED),
    "train_config": TRAIN_CFG,
    "class_count": len(names),
    "classes": names,
}

write_json(RUN_DIRS["config"] / "run_config.json", run_config_payload)

train_args = dict(TRAIN_CFG)
train_args["data"] = str(DATA_YAML_PROCESSED)

write_json(RUN_DIRS["config"] / "train_args_effective.json", train_args)

print("Saved run config:", RUN_DIRS["config"] / "run_config.json")

In [ ]:
# =========================
# Cell 16: Train YOLO26
# =========================

model = YOLO(TRAIN_CFG["model"])

train_results = model.train(**train_args)

train_run_dir = Path(train_results.save_dir)
best_model_path = train_run_dir / "weights" / "best.pt"
last_model_path = train_run_dir / "weights" / "last.pt"

summary = {
    "run_id": RUN_ID,
    "status": "completed",
    "train_run_dir": str(train_run_dir),
    "best_weights": str(best_model_path),
    "last_weights": str(last_model_path),
    "results_csv": str(train_run_dir / "results.csv"),
    "args_yaml": str(train_run_dir / "args.yaml"),
}

write_json(RUN_DIRS["metrics"] / "train_summary.json", summary)

print("Training complete.")
print("Train dir:", train_run_dir)
print("Best weights:", best_model_path)

In [ ]:
# =========================
# Cell 17: Validate on val and test splits
# =========================

if not best_model_path.exists():
    raise FileNotFoundError(f"Missing best model: {best_model_path}")

best_model = YOLO(str(best_model_path))

val_results = best_model.val(
    data=str(DATA_YAML_PROCESSED),
    split="val",
    imgsz=TRAIN_CFG["imgsz"],
    batch=TRAIN_CFG["batch"],
    plots=True,
    save_json=True,
    project=str(RUN_DIRS["models"]),
    name="val_eval",
    exist_ok=True,
    device=0,
)

test_results = best_model.val(
    data=str(DATA_YAML_PROCESSED),
    split="test",
    imgsz=TRAIN_CFG["imgsz"],
    batch=TRAIN_CFG["batch"],
    plots=True,
    save_json=True,
    project=str(RUN_DIRS["models"]),
    name="test_eval",
    exist_ok=True,
    device=0,
)

val_summary = {
    "map50": float(val_results.box.map50),
    "map50_95": float(val_results.box.map),
    "save_dir": str(val_results.save_dir),
}
test_summary = {
    "map50": float(test_results.box.map50),
    "map50_95": float(test_results.box.map),
    "save_dir": str(test_results.save_dir),
}

write_json(RUN_DIRS["metrics"] / "val_summary.json", val_summary)
write_json(RUN_DIRS["metrics"] / "test_summary.json", test_summary)

print("Val:", val_summary)
print("Test:", test_summary)

In [ ]:
# =========================
# Cell 18: Extract per-class metrics
# =========================

def extract_ultralytics_class_metrics(results, class_names):
    box = results.box
    rows = []

    for i, class_name in enumerate(class_names):
        precision = float("nan")
        recall = float("nan")
        map50 = float("nan")
        map50_95 = float("nan")

        try:
            class_result = box.class_result(i)
            if len(class_result) >= 4:
                precision = float(class_result[0])
                recall = float(class_result[1])
                map50 = float(class_result[2])
                map50_95 = float(class_result[3])
        except Exception:
            pass

        rows.append({
            "class_id": i,
            "class": class_name,
            "precision": precision,
            "recall": recall,
            "map50": map50,
            "map50_95": map50_95,
        })

    return pd.DataFrame(rows)


val_metrics_df = extract_ultralytics_class_metrics(val_results, names)
test_metrics_df = extract_ultralytics_class_metrics(test_results, names)

val_metrics_path = RUN_DIRS["metrics"] / "val_per_class_metrics.csv"
test_metrics_path = RUN_DIRS["metrics"] / "test_per_class_metrics.csv"

val_metrics_df.to_csv(val_metrics_path, index=False)
test_metrics_df.to_csv(test_metrics_path, index=False)

display(val_metrics_df)
display(test_metrics_df)

print("Saved:", val_metrics_path)
print("Saved:", test_metrics_path)

In [ ]:
# =========================
# Cell 19: GALVAT custom weighted scoring metric
# =========================

import numpy as np
import pandas as pd
import json
from pathlib import Path

THRESHOLDS = np.array([0.00, 0.05, 0.10, 0.20, 0.50])

# Ratings ordered by error-rate threshold:
# [0%, <=5%, <=10%, <=20%, <=50%]
#
# "fn" = importance that false negatives stay below threshold
# "fp" = importance that false positives stay below threshold
#
# These class names must exactly match your dataset classes.

GALVAT_RATINGS = {
    "bathroom_sign": {
        "fn": [8, 8, 8, 8, 9],
        "fp": [7, 8, 8, 8, 9],
    },
    "canned_good": {
        "fn": [2, 2, 2, 3, 4],
        "fp": [2, 2, 2, 3, 5],
    },
    "cellphone": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [9, 9, 9, 9, 9],
    },
    "chair": {
        "fn": [8, 8, 8, 9, 9],
        "fp": [10, 10, 10, 10, 10],
    },
    "coffee_mug": {
        "fn": [8, 9, 9, 10, 10],
        "fp": [5, 5, 5, 5, 6],
    },
    "curb": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [8, 8, 8, 9, 10],
    },
    "deodorant": {
        "fn": [6, 6, 7, 7, 8],
        "fp": [6, 6, 7, 7, 7],
    },
    "door": {
        "fn": [9, 9, 9, 9, 9],
        "fp": [10, 10, 10, 10, 10],
    },
    "drinking_glass": {
        "fn": [8, 8, 9, 9, 10],
        "fp": [5, 5, 5, 5, 6],
    },
    "electrical_outlet": {
        "fn": [7, 7, 8, 8, 9],
        "fp": [7, 7, 8, 8, 9],
    },
    "exit_sign": {
        "fn": [7, 7, 7, 8, 8],
        "fp": [7, 8, 8, 8, 9],
    },
    "eyeglasses": {
        "fn": [8, 8, 8, 9, 10],
        "fp": [3, 3, 4, 5, 6],
    },
    "fork": {
        "fn": [7, 7, 7, 8, 9],
        "fp": [6, 7, 7, 8, 8],
    },
    "handbag": {
        "fn": [9, 9, 9, 9, 9],
        "fp": [10, 10, 10, 10, 10],
    },
    "handrail": {
        "fn": [8, 8, 8, 9, 9],
        "fp": [7, 7, 8, 8, 9],
    },
    "hearing_aid": {
        "fn": [9, 9, 9, 10, 10],
        "fp": [4, 4, 5, 6, 7],
    },
    "jug": {
        "fn": [6, 6, 6, 7, 8],
        "fp": [5, 5, 6, 6, 7],
    },
    "key": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [8, 8, 8, 8, 8],
    },
    "knife": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [6, 7, 7, 8, 8],
    },
    "lemon": {
        "fn": [4, 4, 5, 6, 7],
        "fp": [4, 4, 5, 6, 7],
    },
    "light_switch": {
        "fn": [7, 7, 8, 8, 9],
        "fp": [5, 5, 5, 5, 6],
    },
    "measuring_cup": {
        "fn": [4, 4, 5, 6, 7],
        "fp": [3, 3, 3, 3, 4],
    },
    "medicine": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [10, 10, 10, 10, 10],
    },
    "paper_towel_dispenser": {
        "fn": [5, 5, 6, 7, 7],
        "fp": [5, 5, 6, 7, 7],
    },
    "pot": {
        "fn": [7, 7, 8, 8, 9],
        "fp": [5, 5, 5, 5, 6],
    },
    "razor": {
        "fn": [8, 8, 8, 9, 9],
        "fp": [6, 7, 7, 8, 8],
    },
    "remote_control": {
        "fn": [4, 4, 5, 6, 7],
        "fp": [3, 3, 3, 4, 5],
    },
    "salt_pepper_shaker": {
        "fn": [6, 6, 6, 7, 8],
        "fp": [2, 2, 2, 3, 4],
    },
    "soap_dispenser": {
        "fn": [6, 6, 7, 7, 8],
        "fp": [7, 7, 7, 8, 8],
    },
    "spoon": {
        "fn": [5, 5, 5, 5, 6],
        "fp": [5, 5, 5, 5, 6],
    },
    "stairs": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [8, 8, 8, 9, 10],
    },
    "table": {
        "fn": [7, 7, 7, 8, 8],
        "fp": [7, 7, 7, 8, 8],
    },
    "thermostat": {
        "fn": [2, 2, 2, 2, 3],
        "fp": [2, 2, 2, 2, 3],
    },
    "toilet": {
        "fn": [9, 9, 9, 10, 10],
        "fp": [10, 10, 10, 10, 10],
    },
    "toothbrush": {
        "fn": [6, 6, 7, 7, 8],
        "fp": [7, 7, 8, 8, 9],
    },
    "toothpaste": {
        "fn": [8, 8, 9, 9, 9],
        "fp": [8, 8, 9, 9, 9],
    },
    "trash_can": {
        "fn": [5, 5, 5, 5, 6],
        "fp": [6, 6, 7, 7, 7],
    },
    "tupperware": {
        "fn": [3, 3, 3, 4, 5],
        "fp": [2, 3, 3, 4, 5],
    },
    "urinal": {
        "fn": [9, 9, 9, 10, 10],
        "fp": [10, 10, 10, 10, 10],
    },
    "wallet": {
        "fn": [9, 9, 9, 9, 9],
        "fp": [10, 10, 10, 10, 10],
    },
    "white_cane": {
        "fn": [10, 10, 10, 10, 10],
        "fp": [8, 8, 8, 8, 8],
    },
    "window": {
        "fn": [4, 4, 4, 5, 5],
        "fp": [7, 8, 8, 8, 8],
    },
    "wine_glass": {
        "fn": [8, 8, 9, 9, 10],
        "fp": [5, 5, 5, 5, 6],
    },
    "writing_device": {
        "fn": [2, 2, 3, 4, 4],
        "fp": [1, 2, 3, 3, 4],
    },
}

missing_ratings = sorted(set(names) - set(GALVAT_RATINGS))
extra_ratings = sorted(set(GALVAT_RATINGS) - set(names))

print("Dataset classes:", len(names))
print("Rating classes:", len(GALVAT_RATINGS))
print("Missing ratings:", missing_ratings)
print("Extra ratings not in dataset:", extra_ratings)

if missing_ratings:
    raise RuntimeError("Some dataset classes are missing GALVAT ratings. Add them before scoring.")


def auc(x, y):
    return np.trapz(np.asarray(y, dtype=float), np.asarray(x, dtype=float))


def remaining_area_score(error_rate, ratings):
    """
    error_rate is 0 to 1.
    ratings are importance ratings at [0%, 5%, 10%, 20%, 50%].
    Returns remaining-area score normalized to [0, 1].
    """
    y = np.asarray(ratings, dtype=float)
    max_area = auc(THRESHOLDS, y)

    error_rate = float(error_rate)

    if error_rate >= 0.50:
        return 0.0, max_area, 0.0

    error_rate = max(0.0, min(error_rate, 0.50))

    xs = [error_rate] + [t for t in THRESHOLDS if t > error_rate]
    xs = np.asarray(xs, dtype=float)
    ys = np.interp(xs, THRESHOLDS, y)

    remaining = auc(xs, ys)
    normalized = remaining / max_area if max_area > 0 else 0.0

    return remaining, max_area, normalized


def galvat_weighted_score_from_df(metrics_df, ratings):
    rows = []
    total_remaining = 0.0
    total_possible = 0.0

    for _, row in metrics_df.iterrows():
        class_name = row["class"]

        if class_name not in ratings:
            continue

        precision = row["precision"]
        recall = row["recall"]

        precision = 0.0 if pd.isna(precision) else float(precision)
        recall = 0.0 if pd.isna(recall) else float(recall)

        precision = max(0.0, min(precision, 1.0))
        recall = max(0.0, min(recall, 1.0))

        false_positive_rate = 1.0 - precision
        false_negative_rate = 1.0 - recall

        fn_remaining, fn_possible, fn_score = remaining_area_score(
            false_negative_rate,
            ratings[class_name]["fn"]
        )

        fp_remaining, fp_possible, fp_score = remaining_area_score(
            false_positive_rate,
            ratings[class_name]["fp"]
        )

        class_remaining = fn_remaining + fp_remaining
        class_possible = fn_possible + fp_possible
        class_score = class_remaining / class_possible if class_possible > 0 else 0.0

        total_remaining += class_remaining
        total_possible += class_possible

        rows.append({
            "class": class_name,
            "precision": precision,
            "recall": recall,
            "map50": float(row["map50"]) if "map50" in row and not pd.isna(row["map50"]) else np.nan,
            "map50_95": float(row["map50_95"]) if "map50_95" in row and not pd.isna(row["map50_95"]) else np.nan,
            "false_positive_rate": false_positive_rate,
            "false_negative_rate": false_negative_rate,
            "fn_score": fn_score,
            "fp_score": fp_score,
            "class_score": class_score,
            "possible_weight": class_possible,
            "earned_weighted_points": class_remaining,
            "lost_weighted_points": class_possible - class_remaining,
        })

    overall_score = total_remaining / total_possible if total_possible > 0 else 0.0

    score_df = pd.DataFrame(rows).sort_values("class_score", ascending=True)
    return overall_score, score_df


val_galvat_score, val_galvat_df = galvat_weighted_score_from_df(val_metrics_df, GALVAT_RATINGS)
test_galvat_score, test_galvat_df = galvat_weighted_score_from_df(test_metrics_df, GALVAT_RATINGS)

val_galvat_df.to_csv(RUN_DIRS["metrics"] / "val_galvat_weighted_scores.csv", index=False)
test_galvat_df.to_csv(RUN_DIRS["metrics"] / "test_galvat_weighted_scores.csv", index=False)

galvat_score_summary = {
    "val_galvat_weighted_score": float(val_galvat_score),
    "test_galvat_weighted_score": float(test_galvat_score),
    "val_map50": float(val_results.box.map50),
    "val_map50_95": float(val_results.box.map),
    "test_map50": float(test_results.box.map50),
    "test_map50_95": float(test_results.box.map),
}

write_json(RUN_DIRS["metrics"] / "galvat_score_summary.json", galvat_score_summary)

print(json.dumps(galvat_score_summary, indent=2))

print("\nWorst validation classes:")
display(val_galvat_df.head(15))

print("\nWorst test classes:")
display(test_galvat_df.head(15))

In [ ]:
# =========================
# Cell 20: Create and save evaluation charts/tables
# =========================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
import json
import shutil

charts_dir = RUN_DIRS["reports"] / "charts"
tables_dir = RUN_DIRS["reports"] / "tables"
charts_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Merge standard metrics + GALVAT metrics
val_full_df = val_metrics_df.merge(
    val_galvat_df[
        [
            "class",
            "false_positive_rate",
            "false_negative_rate",
            "fn_score",
            "fp_score",
            "class_score",
            "possible_weight",
            "earned_weighted_points",
            "lost_weighted_points",
        ]
    ],
    on="class",
    how="left",
)

test_full_df = test_metrics_df.merge(
    test_galvat_df[
        [
            "class",
            "false_positive_rate",
            "false_negative_rate",
            "fn_score",
            "fp_score",
            "class_score",
            "possible_weight",
            "earned_weighted_points",
            "lost_weighted_points",
        ]
    ],
    on="class",
    how="left",
)

val_full_df.to_csv(tables_dir / "val_full_metrics_with_galvat_scores.csv", index=False)
test_full_df.to_csv(tables_dir / "test_full_metrics_with_galvat_scores.csv", index=False)

overall_summary_df = pd.DataFrame(
    [
        {
            "split": "val",
            "map50": float(val_results.box.map50),
            "map50_95": float(val_results.box.map),
            "galvat_weighted_score": float(val_galvat_score),
        },
        {
            "split": "test",
            "map50": float(test_results.box.map50),
            "map50_95": float(test_results.box.map),
            "galvat_weighted_score": float(test_galvat_score),
        },
    ]
)

overall_summary_df.to_csv(tables_dir / "overall_evaluation_summary.csv", index=False)

# Save Excel workbook with all main tables
excel_path = tables_dir / "galvat_evaluation_tables.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    overall_summary_df.to_excel(writer, sheet_name="overall_summary", index=False)
    val_metrics_df.to_excel(writer, sheet_name="val_standard_metrics", index=False)
    test_metrics_df.to_excel(writer, sheet_name="test_standard_metrics", index=False)
    val_galvat_df.to_excel(writer, sheet_name="val_galvat_scores", index=False)
    test_galvat_df.to_excel(writer, sheet_name="test_galvat_scores", index=False)
    val_full_df.to_excel(writer, sheet_name="val_full", index=False)
    test_full_df.to_excel(writer, sheet_name="test_full", index=False)

print("Saved Excel workbook:", excel_path)


def save_horizontal_bar(df, value_col, title, xlabel, filename, ascending=True, top_n=None):
    plot_df = df.copy()

    if top_n is not None:
        plot_df = plot_df.sort_values(value_col, ascending=ascending).head(top_n)
    else:
        plot_df = plot_df.sort_values(value_col, ascending=ascending)

    plt.figure(figsize=(12, max(8, len(plot_df) * 0.28)))
    plt.barh(plot_df["class"], plot_df[value_col])
    plt.xlabel(xlabel)
    plt.ylabel("Class")
    plt.title(title)
    plt.tight_layout()
    out_path = charts_dir / filename
    plt.savefig(out_path, dpi=200)
    plt.show()
    print("Saved:", out_path)


def save_scatter(df, x_col, y_col, title, xlabel, ylabel, filename):
    plot_df = df.copy()

    plt.figure(figsize=(9, 7))
    plt.scatter(plot_df[x_col], plot_df[y_col])

    # Annotate worst 10 GALVAT classes
    worst_classes = set(test_galvat_df.head(10)["class"])

    for _, row in plot_df.iterrows():
        if row["class"] in worst_classes and pd.notna(row[x_col]) and pd.notna(row[y_col]):
            plt.annotate(row["class"], (row[x_col], row[y_col]), fontsize=8)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = charts_dir / filename
    plt.savefig(out_path, dpi=200)
    plt.show()
    print("Saved:", out_path)


# Main GALVAT score charts
save_horizontal_bar(
    test_galvat_df,
    value_col="class_score",
    title="Test GALVAT class scores",
    xlabel="GALVAT class score",
    filename="test_galvat_class_scores.png",
    ascending=True,
)

save_horizontal_bar(
    test_galvat_df,
    value_col="lost_weighted_points",
    title="Test lost weighted points by class",
    xlabel="Lost weighted points",
    filename="test_galvat_lost_weighted_points.png",
    ascending=False,
    top_n=20,
)

# Standard metric charts
save_horizontal_bar(
    test_full_df,
    value_col="precision",
    title="Test precision by class",
    xlabel="Precision",
    filename="test_precision_by_class.png",
    ascending=True,
)

save_horizontal_bar(
    test_full_df,
    value_col="recall",
    title="Test recall by class",
    xlabel="Recall",
    filename="test_recall_by_class.png",
    ascending=True,
)

save_horizontal_bar(
    test_full_df,
    value_col="map50_95",
    title="Test mAP50-95 by class",
    xlabel="mAP50-95",
    filename="test_map50_95_by_class.png",
    ascending=True,
)

save_scatter(
    test_full_df,
    x_col="precision",
    y_col="recall",
    title="Test precision vs recall by class",
    xlabel="Precision",
    ylabel="Recall",
    filename="test_precision_vs_recall.png",
)

# Training curves from results.csv
results_csv = train_run_dir / "results.csv"

if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    results_df.columns = [c.strip() for c in results_df.columns]

    results_df.to_csv(tables_dir / "training_results.csv", index=False)

    loss_cols = [c for c in results_df.columns if "loss" in c.lower()]
    metric_cols = [
        c for c in results_df.columns
        if any(k in c.lower() for k in ["map50", "precision", "recall"])
    ]

    if "epoch" in results_df.columns and loss_cols:
        plt.figure(figsize=(12, 7))
        for col in loss_cols:
            plt.plot(results_df["epoch"], results_df[col], label=col)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training and validation losses")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        out_path = charts_dir / "training_validation_losses.png"
        plt.savefig(out_path, dpi=200)
        plt.show()
        print("Saved:", out_path)

    if "epoch" in results_df.columns and metric_cols:
        plt.figure(figsize=(12, 7))
        for col in metric_cols:
            plt.plot(results_df["epoch"], results_df[col], label=col)
        plt.xlabel("Epoch")
        plt.ylabel("Metric")
        plt.title("Training metrics over epochs")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        out_path = charts_dir / "training_metrics_over_epochs.png"
        plt.savefig(out_path, dpi=200)
        plt.show()
        print("Saved:", out_path)

else:
    print("No results.csv found at:", results_csv)

# Copy common Ultralytics-generated plots if they exist
for source_dir in [train_run_dir, Path(val_results.save_dir), Path(test_results.save_dir)]:
    if source_dir.exists():
        for src in source_dir.rglob("*"):
            if src.is_file() and src.suffix.lower() in {".png", ".jpg", ".jpeg", ".pdf", ".csv"}:
                dst = charts_dir / source_dir.name / src.relative_to(source_dir)
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)

print("Saved all charts to:", charts_dir)
print("Saved all tables to:", tables_dir)

In [ ]:
# =========================
# Cell 21: Export ONNX and save every artifact into the Drive run folder
# =========================

from pathlib import Path
import shutil
import json
import os
from ultralytics import YOLO

# Make sure output dirs exist
RUN_DIRS["models"].mkdir(parents=True, exist_ok=True)
RUN_DIRS["exports"].mkdir(parents=True, exist_ok=True)
RUN_DIRS["reports"].mkdir(parents=True, exist_ok=True)
RUN_DIRS["analytics"].mkdir(parents=True, exist_ok=True)
RUN_DIRS["config"].mkdir(parents=True, exist_ok=True)

# -------------------------
# Copy best.pt and last.pt
# -------------------------
if not best_model_path.exists():
    raise FileNotFoundError(f"best.pt not found at {best_model_path}")

best_pt_drive = RUN_DIRS["models"] / "best.pt"
shutil.copy2(best_model_path, best_pt_drive)

last_pt_drive = None
if last_model_path.exists():
    last_pt_drive = RUN_DIRS["models"] / "last.pt"
    shutil.copy2(last_model_path, last_pt_drive)

print("Saved best.pt to:", best_pt_drive)
if last_pt_drive:
    print("Saved last.pt to:", last_pt_drive)

# -------------------------
# Export best.pt to ONNX
# -------------------------
print("\nExporting best.pt to ONNX...")

export_model = YOLO(str(best_model_path))

onnx_export_result = export_model.export(
    format="onnx",
    imgsz=TRAIN_CFG["imgsz"],
    simplify=True,
    opset=12,
    device=0,
)

onnx_path = Path(onnx_export_result)

if not onnx_path.exists():
    fallback = best_model_path.with_suffix(".onnx")
    if fallback.exists():
        onnx_path = fallback
    else:
        raise FileNotFoundError("ONNX export completed but best.onnx was not found.")

best_onnx_drive = RUN_DIRS["exports"] / "best.onnx"
shutil.copy2(onnx_path, best_onnx_drive)

print("Saved best.onnx to:", best_onnx_drive)

# -------------------------
# Hash important artifacts
# -------------------------
artifact_hashes = {}

for artifact in [best_pt_drive, best_onnx_drive]:
    if artifact.exists():
        artifact_hashes[artifact.name] = sha256_file(artifact)

if last_pt_drive and last_pt_drive.exists():
    artifact_hashes[last_pt_drive.name] = sha256_file(last_pt_drive)

write_json(RUN_DIRS["exports"] / "artifact_hashes.json", artifact_hashes)

# -------------------------
# Copy config/data info
# -------------------------
try:
    shutil.copy2(DATA_YAML_PROCESSED, RUN_DIRS["config"] / "data.yaml")
except Exception as e:
    print("Could not copy data.yaml:", e)

with open(RUN_DIRS["config"] / "classes.txt", "w") as f:
    for i, name in enumerate(names):
        f.write(f"{i}: {name}\n")

# -------------------------
# Copy full Ultralytics train/val/test run dirs into analytics
# -------------------------
for source_dir in [
    train_run_dir,
    Path(val_results.save_dir),
    Path(test_results.save_dir),
]:
    if source_dir.exists():
        dst = RUN_DIRS["analytics"] / source_dir.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(source_dir, dst)
        print("Copied analytics folder:", source_dir, "->", dst)

# -------------------------
# Final summary
# -------------------------
final_summary = {
    "run_id": RUN_ID,
    "run_root": str(RUN_ROOT),
    "class_count": len(names),
    "classes": names,

    "best_pt": str(best_pt_drive),
    "best_onnx": str(best_onnx_drive),

    "val_map50": float(val_results.box.map50),
    "val_map50_95": float(val_results.box.map),
    "val_galvat_weighted_score": float(val_galvat_score),

    "test_map50": float(test_results.box.map50),
    "test_map50_95": float(test_results.box.map),
    "test_galvat_weighted_score": float(test_galvat_score),

    "reports_dir": str(RUN_DIRS["reports"]),
    "metrics_dir": str(RUN_DIRS["metrics"]),
    "analytics_dir": str(RUN_DIRS["analytics"]),
    "models_dir": str(RUN_DIRS["models"]),
    "exports_dir": str(RUN_DIRS["exports"]),

    "artifact_hashes": artifact_hashes,
}

write_json(RUN_DIRS["reports"] / "final_summary.json", final_summary)

# Also save a CSV summary
pd.DataFrame([final_summary]).to_csv(RUN_DIRS["reports"] / "final_summary.csv", index=False)

# -------------------------
# Zip the full run folder
# -------------------------
zip_path = shutil.make_archive(
    str(RUN_ROOT),
    "zip",
    root_dir=RUN_ROOT
)

print("\nDONE.")
print("Run folder:", RUN_ROOT)
print("Run zip:", zip_path)
print("best.pt:", best_pt_drive)
print("best.onnx:", best_onnx_drive)
print("Final summary:", RUN_DIRS["reports"] / "final_summary.json")

In [ ]:
# =========================
# Cell 22B: Zip processed dataset and save to Drive
# =========================

from pathlib import Path
import shutil
from datetime import datetime

# Source processed dataset folder
PROCESSED_DIR = Path("/content/final_dataset_processed_640")

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(f"Processed dataset folder not found: {PROCESSED_DIR}")

# Drive destination folder
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DRIVE_DATASET_BACKUP_DIR = Path("/content/drive/MyDrive/galvat_processed_dataset_backups")
DRIVE_DATASET_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Create zip locally first
local_zip_base = Path(f"/content/final_dataset_processed_640_{timestamp}")
local_zip_path = shutil.make_archive(
    base_name=str(local_zip_base),
    format="zip",
    root_dir=PROCESSED_DIR.parent,
    base_dir=PROCESSED_DIR.name,
)

local_zip_path = Path(local_zip_path)

print("Created local zip:")
print(local_zip_path)
print("Zip size GB:", round(local_zip_path.stat().st_size / 1_000_000_000, 3))

# Copy zip to Drive
drive_zip_path = DRIVE_DATASET_BACKUP_DIR / local_zip_path.name
shutil.copy2(local_zip_path, drive_zip_path)

print("\nSaved processed dataset zip to Drive:")
print(drive_zip_path)
print("Drive zip size GB:", round(drive_zip_path.stat().st_size / 1_000_000_000, 3))